# Tiny Transformer · Rank-3 Attention Scores

Merging the batch and head axes preserved FP32 scores and logits across eight held-out inputs. On SM6650, both W8A16 variants passed the device numeric check. Six paired captures showed a 1.62% lower pooled p50, within the observed 17.27% A/A p50 variation. The speedup did not meet the acceptance gate.

This notebook reproduces the FP32 check and reads the saved device experiment. See the [experiment protocol](../docs/experiments/claude-code-attention-layout.md) for the evidence that motivated the change and the QAIRT steps used to collect the trace.

## Export both layouts and verify the graph

Run from the repository with the project's Python environment. The held-out inputs come from the Android Tiny experiment workspace, separate from its 48 calibration samples. The candidate reuses the baseline weights and changes only the input layout of the score MatMul, from `[B,H,T,D] × [B,H,D,T]` to `[B×H,T,D] × [B×H,D,T]`. Scores regain their original shape before scaling. Diagnostic ONNX copies expose the unscaled scores and leave the registered ONNX exports unchanged.

In [ ]:
import json

from IPython.display import Markdown, display

from htp_insight.core import Core
from scripts.prepare_tiny_attention_layout import main as prepare_layout

workspace = Core().root
device_records = workspace / "attention-score-layout"
prepare_layout()
fp32 = json.loads((workspace / "tiny-attention-layout-fp32.json").read_text())
print("Score MatMul input ranks", fp32["input_ranks"])
for artifact in fp32["models"]:
    print(artifact["id"], artifact["name"])

## Inspect FP32 error

The script compares unscaled scores in ONNX Runtime, final logits in PyTorch and ONNX Runtime, and each export against its PyTorch reference. Each check uses `atol=1e-5` and `rtol=1e-4`.

In [ ]:
checks = fp32["fp32"]
assert len(checks) == 8
rows = ["| Check | Largest absolute error | Highest sample mean error | Worst input |",
        "| --- | ---: | ---: | --- |"]
for check in checks[0]:
    outputs = [next(iter(sample[check]["outputs"].values())) for sample in checks]
    worst = max(range(len(outputs)), key=lambda index: outputs[index]["max_abs"])
    highest_mean = max(output["mean_abs"] for output in outputs)
    rows.append(f"| {check} | {outputs[worst]['max_abs']:.8g} | {highest_mean:.8g} | {worst} (zero-based) |")
display(Markdown("\n".join(rows)))

## Check saved device outputs and paired latency

The cell reads `numeric.json` and `paired-results.json` from `runs/attention-score-layout/`. To create new device records, prepare the matched W8A16 artifacts as described in the experiment protocol and run `.venv/bin/python -m scripts.run_tiny_attention_ab --sdk <QAIRT-directory>` with a supported device. The three A/A blocks establish the noise reference. The six A/B blocks alternate execution order and keep 100 Basic samples per side.

In [ ]:
numeric_file = device_records / "numeric.json"
paired_file = device_records / "paired-results.json"
paired = json.loads(paired_file.read_text()) if paired_file.is_file() else None
if paired and paired.get("comparison_issues"):
    print("The saved captures report incompatible conditions:", paired["comparison_issues"])
    paired = None
elif paired is not None and numeric_file.is_file():
    device_checks = json.loads(numeric_file.read_text())
    assert len(device_checks) == 8 and len(paired["aa"]) == 3 and len(paired["ab"]) == 6
    maximum = max(check["outputs"]["logits"]["max_abs"]
                  for sample in device_checks for check in sample["checks"])
    p50_change, p95_change = paired["delta_pct"]["p50"], paired["delta_pct"]["p95"]
    p50_noise, p95_noise = paired["aa_max_delta_pct"]["p50"], paired["aa_max_delta_pct"]["p95"]
    (base_p50, base_p95), (rank3_p50, rank3_p95) = paired["percentiles_us"]
    display(Markdown(f"Device numeric gate **{'passed' if paired['numeric_passed'] else 'failed'}** "
                     f"(maximum absolute error {maximum:.5f}). "
                     f"Performance gate **{'passed' if paired['performance_passed'] else 'failed'}**."))
    rows = ["| NetRun metric | Baseline (µs) | Rank-3 (µs) | A/B change | Max A/A variation |",
            "| --- | ---: | ---: | ---: | ---: |",
            f"| Pooled p50 | {base_p50:.2f} | {rank3_p50:.2f} | {p50_change:+.2f}% | {p50_noise:.2f}% |",
            f"| Pooled p95 | {base_p95:.2f} | {rank3_p95:.2f} | {p95_change:+.2f}% | {p95_noise:.2f}% |"]
    display(Markdown("\n".join(rows)))
    print(f"Rank-3 p50 wins {paired['wins_p50']}/{len(paired['ab'])}.")
else:
    paired = None
    print("Saved device evidence is unavailable. The FP32 checks above ran independently.")

### Paired p50 by block

Each bar measures the percentage change from the baseline within that block. A negative value means the rank-3 capture finished faster.

In [ ]:
if paired is not None:
    blocks = []
    for number, (baseline, candidate) in enumerate(paired["ab"], 1):
        change = 100 * (candidate["latency_us"]["p50"] / baseline["latency_us"]["p50"] - 1)
        blocks.append((number, change))
    extent = max(abs(change) for _, change in blocks)
    center = 250
    scale = 130 / max(extent, 1)
    svg = ['<svg xmlns="http://www.w3.org/2000/svg" viewBox="0 0 440 216" width="440" '
           'style="max-width:100%;font:13px system-ui;background:#fff;color:#17202b" role="img" '
           'aria-label="Six paired NetRun p50 percentage changes">',
           '<line x1="250" x2="250" y1="30" y2="205" stroke="#667584"/>',
           '<text x="250" y="20" text-anchor="middle" fill="currentColor">0%</text>']
    for number, change in blocks:
        y = 26 + number * 26
        signed_width = change * scale
        start = center + min(0, signed_width)
        fill = "#196a88" if change < 0 else "#a25222"
        svg.append(f'<text x="12" y="{y + 4}" fill="currentColor">Block {number}</text>')
        svg.append(f'<rect x="{start}" y="{y - 7}" width="{abs(signed_width)}" height="14" fill="{fill}"/>')
        x = center + signed_width + (-5 if change < 0 else 5)
        anchor = "end" if change < 0 else "start"
        svg.append(f'<text x="{x}" y="{y + 4}" text-anchor="{anchor}" fill="currentColor">{change:+.1f}%</text>')
    from IPython.display import HTML
    display(HTML("".join([*svg, "</svg>"])))

## Interpret the run

The rank-3 layout passed the correctness gate. In the separately acquired Optrace, operations mapped to the score MatMul went from 69,534 to 63,877 summed cycles. The Basic captures did not establish an end-to-end speedup under the protocol's p50, p95, and repeatability criteria. Trace cycle sums capture operation work, including parallel work, and are reported separately from NetRun wall time.